# Pesquisa de Preços — protótipo (PGE-AM / Coordenadoria de Transparência e Ouvidoria)

Ferramenta de apoio à pesquisa de mercado da Gerência de Material, a partir dos **Preços Praticados** publicados em dados abertos pelo Compras.gov.br (SEGES/MGI).

**Protótipo**, em uso das competências institucionais da PGE; não substitui o juízo do pregoeiro/agente de contratação. A metodologia estatística segue a **IN SEGES/ME nº 65/2021** (média, mediana, menor preço e saneamento de preços inexequíveis ou excessivos).

Como usar: preencha a célula 1 com o código do item e rode **Ambiente de execução → Executar tudo**. O código CATMAT (material) ou CATSER (serviço) é encontrado em https://catalogo.compras.gov.br .

In [ ]:
# 1. O que pesquisar
TIPO = 'material'          # 'material' (CATMAT) ou 'servico' (CATSER)
CODIGO_CATALOGO = 341842   # código do item no catálogo

# Filtros opcionais (deixe '' para não filtrar)
DATA_INICIO = '2024-01-01' # período das compras consideradas
DATA_FIM    = '2026-11-01'
ESTADO      = ''           # ex.: 'AM' para compras no Amazonas; vazio = Brasil todo
ESFERA      = ''           # 'F' Federal, 'E' Estadual, 'M' Municipal
PODER       = ''           # 'E' Executivo, 'L' Legislativo, 'J' Judiciário
UNIDADE_FORNECIMENTO = ''  # ex.: 'UNIDADE', 'PACOTE' — filtra para comparar o mesmo tipo de embalagem


In [ ]:
# 2. Baixar os preços praticados (todas as páginas)
import urllib.request, urllib.parse, json, statistics as st
from datetime import date

BASE = 'https://dadosabertos.compras.gov.br'
EP = {'material': '/modulo-pesquisa-preco/1_consultarMaterial',
      'servico':  '/modulo-pesquisa-preco/3_consultarServico'}[TIPO]
CHAVE = 'codigo' if TIPO == 'material' else 'codigoItemCatalogo'

def consultar():
    reg, pagina = [], 1
    while True:
        q = {'pagina': pagina, 'tamanhoPagina': 500,
             'dataCompraInicio': DATA_INICIO, 'dataCompraFim': DATA_FIM}
        if TIPO == 'material': q['tipo'] = 'codigoItemCatalogo'
        q[CHAVE] = CODIGO_CATALOGO
        for k, v in [('estado', ESTADO), ('esfera', ESFERA), ('poder', PODER)]:
            if v: q[k] = v
        url = f'{BASE}{EP}?' + urllib.parse.urlencode({k: v for k, v in q.items() if v != ''})
        req = urllib.request.Request(url, headers={'Accept': 'application/json'})
        with urllib.request.urlopen(req, timeout=90) as r:
            j = json.loads(r.read())
        reg += j.get('resultado', [])
        if pagina >= j.get('totalPaginas', 1): break
        pagina += 1
    return reg

registros = consultar()
if UNIDADE_FORNECIMENTO:
    registros = [r for r in registros if (r.get('nomeUnidadeFornecimento') or '').upper() == UNIDADE_FORNECIMENTO.upper()]
print(f'{len(registros)} compras encontradas para o item {CODIGO_CATALOGO}.')
if registros:
    print('Item:', registros[0].get('descricaoItem', '')[:120])


In [ ]:
# 3. Estatística e saneamento (IN 65/2021)
precos = sorted(r['precoUnitario'] for r in registros if (r.get('precoUnitario') or 0) > 0)
assert precos, 'Nenhum preço válido — reveja o código do item ou amplie o período.'

def resumo(v):
    media = st.mean(v); mediana = st.median(v)
    dp = st.pstdev(v) if len(v) > 1 else 0.0
    cv = dp / media * 100 if media else 0
    return dict(n=len(v), menor=min(v), maior=max(v), media=media, mediana=mediana, desvio=dp, cv=cv)

bruto = resumo(precos)
# Saneamento: descarta preços fora de média ± 1 desvio-padrão (critério usual p/ inexequíveis/excessivos)
li, ls = bruto['media'] - bruto['desvio'], bruto['media'] + bruto['desvio']
saneado = [p for p in precos if li <= p <= ls] or precos
san = resumo(saneado)
descartados = len(precos) - len(saneado)

ref = san['mediana'] if san['cv'] > 25 else san['media']  # alta dispersão -> mediana
brl = lambda x: 'R$ ' + f'{x:,.4f}'.replace(',', '§').replace('.', ',').replace('§', '.')
print(f"Amostra: {bruto['n']} preços | média {brl(bruto['media'])} | mediana {brl(bruto['mediana'])} | CV {bruto['cv']:.1f}%")
print(f"Saneada: {san['n']} preços ({descartados} descartados) | média {brl(san['media'])} | mediana {brl(san['mediana'])} | CV {san['cv']:.1f}%")
print(f"PREÇO DE REFERÊNCIA sugerido: {brl(ref)}  ({'mediana — alta dispersão' if san['cv']>25 else 'média'})")


In [ ]:
# 4. Planilha dos preços (CSV, pronta para anexar ao processo)
import csv
cols = [('dataResultado','Data'),('precoUnitario','Preço unitário'),('nomeUnidadeFornecimento','Unidade'),
        ('marca','Marca'),('nomeFornecedor','Fornecedor'),('nomeOrgao','Órgão comprador'),
        ('estado','UF'),('municipio','Município'),('modalidade','Modalidade'),('quantidade','Qtd'),('idCompra','Compra')]
arq = f'pesquisa_precos_{CODIGO_CATALOGO}.csv'
with open(arq, 'w', newline='', encoding='utf-8-sig') as f:
    w = csv.writer(f, delimiter=';'); w.writerow([c[1] for c in cols])
    for r in sorted(registros, key=lambda x: x.get('precoUnitario', 0)):
        w.writerow([r.get(c[0], '') for c in cols])
print('CSV gerado:', arq)
try:
    from google.colab import files; files.download(arq)
except Exception: pass


In [ ]:
# 5. Relatório de pesquisa de preços (HTML, 1 página)
import html
desc = html.escape(registros[0].get('descricaoItem','')) if registros else ''
linhas = ''.join(
    f"<tr><td>{html.escape(str(r.get('dataResultado','')))}</td><td class=p>{brl(r['precoUnitario'])}</td>"
    f"<td>{html.escape(str(r.get('nomeUnidadeFornecimento','')))}</td><td>{html.escape(str(r.get('marca','') or ''))}</td>"
    f"<td>{html.escape(str(r.get('nomeFornecedor','')))}</td><td>{html.escape(str(r.get('nomeOrgao','')))}</td>"
    f"<td>{html.escape(str(r.get('estado','')))}</td></tr>"
    for r in sorted(registros, key=lambda x: x.get('precoUnitario',0)) if r.get('precoUnitario'))
rel = f'''<!DOCTYPE html><html lang=pt-BR><head><meta charset=utf-8>
<style>body{{font-family:Arial;margin:30px;color:#222;font-size:13px}}h1{{font-size:17px}}
table{{border-collapse:collapse;width:100%;margin-top:8px}}th,td{{border:1px solid #ccc;padding:4px 7px;text-align:left}}
th{{background:#12239E;color:#fff}}.p{{text-align:right;font-variant-numeric:tabular-nums}}
.cards{{display:flex;gap:14px;margin:12px 0}}.c{{border:1px solid #ddd;padding:8px 14px;border-radius:6px}}
.c b{{display:block;font-size:18px;color:#12239E}}.ref{{background:#DEECF9}}.nota{{color:#555;font-size:11px;margin-top:14px}}</style></head><body>
<h1>Pesquisa de Preços — item {CODIGO_CATALOGO}</h1>
<div><b>{desc}</b></div>
<div class=cards>
<div class=c>Menor preço<b>{brl(san['menor'])}</b></div>
<div class=c>Média (saneada)<b>{brl(san['media'])}</b></div>
<div class=c>Mediana (saneada)<b>{brl(san['mediana'])}</b></div>
<div class='c ref'>Preço de referência<b>{brl(ref)}</b></div>
<div class=c>Preços usados<b>{san['n']} de {bruto['n']}</b></div></div>
<table><thead><tr><th>Data</th><th>Preço unitário</th><th>Unidade</th><th>Marca</th><th>Fornecedor</th><th>Órgão comprador</th><th>UF</th></tr></thead>
<tbody>{linhas}</tbody></table>
<p class=nota>Fonte: Compras.gov.br — Preços Praticados (dados abertos), período {DATA_INICIO} a {DATA_FIM}. Metodologia: IN SEGES/ME nº 65/2021 — saneamento de preços fora de média ± 1 desvio-padrão; referência pela mediana quando o coeficiente de variação supera 25%. Coeficiente de variação da amostra saneada: {san['cv']:.1f}%. Protótipo de apoio; a decisão cabe ao agente de contratação.</p>
</body></html>'''
arqr = f'relatorio_precos_{CODIGO_CATALOGO}.html'
open(arqr, 'w', encoding='utf-8').write(rel)
print('Relatório gerado:', arqr)
try:
    from google.colab import files; files.download(arqr)
except Exception: pass
